In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, classification_report

pd.set_option("display.max_columns", 120)
DATA_DIR = "../data/"

model_df = pd.read_parquet(DATA_DIR + "train_features.parquet")
print("Loaded:", model_df.shape)

Loaded: (1122452, 214)


In [3]:
counter_cols = ["171_0", "666_0", "427_0", "837_0", "309_0", "835_0", "370_0", "100_0"]
rate_cols    = [f"{c}_rate"  for c in counter_cols]
reset_cols   = [f"{c}_reset" for c in counter_cols]
share_cols   = [c for c in model_df.columns if c.endswith("_share")]
spec_cols    = [c for c in model_df.columns if c.startswith("Spec_")]

feature_cols = counter_cols + rate_cols + reset_cols + share_cols + spec_cols + ["time_step"]
print("Features:", len(feature_cols))

Features: 212


In [4]:
rng = np.random.default_rng(42)
vehicles = model_df["vehicle_id"].unique()
rng.shuffle(vehicles)

n_val = int(0.2 * len(vehicles))
val_vehicles   = set(vehicles[:n_val])
train_vehicles = set(vehicles[n_val:])

is_val = model_df["vehicle_id"].isin(val_vehicles)
train_df, val_df = model_df[~is_val], model_df[is_val]

print(f"Vehicles  - train: {len(train_vehicles):,}  val: {len(val_vehicles):,}")
print(f"Readouts  - train: {len(train_df):,}  val: {len(val_df):,}")
print("\nVal class distribution:")
print(val_df["class_label"].value_counts().sort_index())

Vehicles  - train: 18,840  val: 4,710
Readouts  - train: 898,531  val: 223,921

Val class distribution:
class_label
0    218816
1      2530
2      1220
3       596
4       759
Name: count, dtype: int64


In [5]:
X_train = train_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y_train = train_df["class_label"].values

X_val   = val_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y_val   = val_df["class_label"].values

print("X_train:", X_train.shape, " X_val:", X_val.shape)
print("Any NaN left?", X_train.isna().sum().sum(), X_val.isna().sum().sum())

X_train: (898531, 212)  X_val: (223921, 212)
Any NaN left? 0 0


In [6]:
COST = np.array([
    [  0,   7,   8,   9,  10],   # actual 0
    [200,   0,   7,   8,   9],   # actual 1
    [300, 200,   0,   7,   8],   # actual 2
    [400, 300, 200,   0,   7],   # actual 3
    [500, 400, 300, 200,   0],   # actual 4
])

def total_cost(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3, 4])
    return int((cm * COST).sum())

def cost_report(y_true, y_pred, name):
    tc = total_cost(y_true, y_pred)
    print(f"{name:<28} Total_cost = {tc:>10,}   (per readout: {tc/len(y_true):.2f})")
    return tc

In [7]:
always_0 = np.zeros_like(y_val)
always_4 = np.full_like(y_val, 4)

cost_report(y_val, always_0, "Always predict 0 (healthy)")
cost_report(y_val, always_4, "Always predict 4 (alarm all)")

Always predict 0 (healthy)   Total_cost =  1,489,900   (per readout: 6.65)
Always predict 4 (alarm all) Total_cost =  2,224,862   (per readout: 9.94)


2224862

In [ ]:
from sklearn.linear_model import SGDClassifier

clf = Pipeline([
    ("scale", StandardScaler()),
    ("lr", SGDClassifier(
        loss="log_loss",          # logistic regression via stochastic gradient descent
        class_weight="balanced",
        max_iter=30,
        tol=1e-3,
        random_state=42,
    )),
])

clf.fit(X_train, y_train)
print("Trained.")

Trained.


d:\AIPM_Bootcamp\CAE_Projects\engineering-ai-digital-twin-predictive-maintenance\venv\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:741: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


In [19]:
X_train.head(3)

,171_0,666_0,427_0,837_0,309_0,835_0,370_0,100_0,171_0_rate,666_0_rate,427_0_rate,837_0_rate,309_0_rate,835_0_rate,370_0_rate,100_0_rate,171_0_reset,666_0_reset,427_0_reset,837_0_reset,309_0_reset,835_0_reset,370_0_reset,100_0_reset,167_0_share,167_1_share,167_2_share,167_3_share,167_4_share,167_5_share,167_6_share,167_7_share,167_8_share,167_9_share,272_0_share,272_1_share,272_2_share,272_3_share,272_4_share,272_5_share,272_6_share,272_7_share,272_8_share,272_9_share,291_0_share,291_1_share,291_2_share,291_3_share,291_4_share,291_5_share,291_6_share,291_7_share,291_8_share,291_9_share,291_10_share,158_0_share,158_1_share,158_2_share,158_3_share,158_4_share,...,Spec_1=Cat9,Spec_2=Cat0,Spec_2=Cat1,Spec_2=Cat10,Spec_2=Cat11,Spec_2=Cat12,Spec_2=Cat13,Spec_2=Cat14,Spec_2=Cat15,Spec_2=Cat16,Spec_2=Cat17,Spec_2=Cat18,Spec_2=Cat19,Spec_2=Cat2,Spec_2=Cat20,Spec_2=Cat3,Spec_2=Cat4,Spec_2=Cat5,Spec_2=Cat6,Spec_2=Cat7,Spec_2=Cat8,Spec_2=Cat9,Spec_3=Cat0,Spec_3=Cat1,Spec_3=Cat2,Spec_3=Cat3,Spec_4=Cat0,Spec_4=Cat1,Spec_5=Cat0,Spec_5=Cat1,Spec_5=Cat2,Spec_5=Cat3,Spec_5=Cat4,Spec_6=Cat0,Spec_6=Cat1,Spec_6=Cat10,Spec_6=Cat11,Spec_6=Cat12,Spec_6=Cat13,Spec_6=Cat15,Spec_6=Cat17,Spec_6=Cat18,Spec_6=Cat2,Spec_6=Cat3,Spec_6=Cat4,Spec_6=Cat5,Spec_6=Cat6,Spec_6=Cat7,Spec_6=Cat8,Spec_6=Cat9,Spec_7=Cat0,Spec_7=Cat1,Spec_7=Cat2,Spec_7=Cat3,Spec_7=Cat4,Spec_7=Cat5,Spec_7=Cat6,Spec_7=Cat7,Spec_7=Cat8,time_step
0,167985.0,10787.0,7413813.0,2296.0,70.0,8036751.0,0.0,858410.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0,0,0,0,0,0,0,0,0.000328,0.103409,0.129879,0.050280,0.101264,0.380715,0.215901,0.017726,0.000498,0.0,0.130602,0.078053,0.034999,0.060851,0.658875,0.036265,0.000354,0.0,0.0,0.0,0.197871,0.089502,0.074665,0.149169,0.075472,0.036284,0.086276,0.083212,0.079342,0.117562,0.010643,0.009569,0.265214,0.290376,0.077733,0.067779,...,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,11.2
1,167985.0,10787.0,7413813.0,2296.0,70.0,8040811.0,0.0,860571.0,0.000000,0.000000,0.000000,0.000000,0.0,20300.000000,0.0,10805.000000,0,0,0,0,0,0,0,0,0.000328,0.103869,0.129812,0.050254,0.101212,0.380520,0.215790,0.017717,0.000497,0.0,0.131043,0.078013,0.034982,0.060820,0.658541,0.036247,0.000354,0.0,0.0,0.0,0.198068,0.089855,0.074557,0.148953,0.075523,0.036393,0.086151,0.083092,0.079388,0.117391,0.010628,0.009564,0.265380,0.290436,0.077693,0.067745,...,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,11.4
2,331635.0,14525.0,13683604.0,2600.0,70.0,12777022.0,0.0,1379191.0,19957.317073,455.853659,764608.658537,37.073171,0.0,577586.707317,0.0,63246.341463,0,0,0,0,0,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.102363,0.064882,0.034261,0.066824,0.705095,0.026353,0.000223,0.0,0.0,0.0,0.215084,0.096063,0.085591,0.142987,0.072702,0.041486,0.088611,0.067063,0.059007,0.115094,0.016313,0.011267,0.263567,0.287227,0.095473,0.088155,...,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,19.6


In [26]:
print(y_train[10000:10010])

[0 0 0 0 0 0 0 0 0 0]


In [27]:
train_df["class_label"]

0          0
1          0
2          0
3          0
4          0
          ..
1122447    0
1122448    0
1122449    0
1122450    0
1122451    0
Name: class_label, Length: 898531, dtype: int64

In [14]:
pred_argmax = clf.predict(X_val)

cost_report(y_val, pred_argmax, "SDGC (argmax)")

print("\nConfusion matrix (rows=actual, cols=predicted):")
print(pd.DataFrame(confusion_matrix(y_val, pred_argmax, labels=[0,1,2,3,4]),
                   index=[f"actual {i}" for i in range(5)],
                   columns=[f"pred {i}" for i in range(5)]))

print("\n", classification_report(y_val, pred_argmax, zero_division=0))


SDGC (argmax)                Total_cost =  1,359,692   (per readout: 6.07)

Confusion matrix (rows=actual, cols=predicted):
          pred 0  pred 1  pred 2  pred 3  pred 4
actual 0  206281    2079    5453    3248    1755
actual 1    2111      54     206     130      29
actual 2     969      44      98      80      29
actual 3     461      16      59      46      14
actual 4     567      34      67      56      35

               precision    recall  f1-score   support

           0       0.98      0.94      0.96    218816
           1       0.02      0.02      0.02      2530
           2       0.02      0.08      0.03      1220
           3       0.01      0.08      0.02       596
           4       0.02      0.05      0.03       759

    accuracy                           0.92    223921
   macro avg       0.21      0.23      0.21    223921
weighted avg       0.96      0.92      0.94    223921



In [15]:
proba = clf.predict_proba(X_val)          # (n_samples, 5)

# Expected cost of predicting class m = sum over actual classes n of P(n) * COST[n, m]
expected_cost = proba @ COST              # (n_samples, 5)
pred_mincost  = expected_cost.argmin(axis=1)

cost_report(y_val, pred_mincost, "SDGC (min expected cost)")

print("\nPrediction distribution:")
print(pd.Series(pred_mincost).value_counts().sort_index())

SDGC (min expected cost)     Total_cost =  2,014,938   (per readout: 9.00)

Prediction distribution:
0     20625
1       998
2     10129
3     17573
4    174596
Name: count, dtype: int64


In [16]:
# Are the probabilities calibrated? Compare predicted mean vs actual frequency.
print("Predicted mean probability per class:")
print(pd.Series(proba.mean(axis=0), index=range(5)).round(4))
print("\nActual class frequency in validation:")
print((pd.Series(y_val).value_counts(normalize=True).sort_index()).round(4))

Predicted mean probability per class:
0    0.6924
1    0.0977
2    0.0836
3    0.0608
4    0.0654
dtype: float64

Actual class frequency in validation:
0    0.9772
1    0.0113
2    0.0054
3    0.0027
4    0.0034
Name: proportion, dtype: float64


In [17]:
clf_unw = Pipeline([
    ("scale", StandardScaler()),
    ("lr", SGDClassifier(loss="log_loss", max_iter=50, tol=1e-3,
                         early_stopping=True, random_state=42)),   # no class_weight
])
clf_unw.fit(X_train, y_train)

proba_unw = clf_unw.predict_proba(X_val)
pred_unw_argmax  = proba_unw.argmax(axis=1)
pred_unw_mincost = (proba_unw @ COST).argmin(axis=1)

cost_report(y_val, pred_unw_argmax,  "SGD unweighted (argmax)")
cost_report(y_val, pred_unw_mincost, "SGD unweighted (min cost)")
print(pd.Series(pred_unw_mincost).value_counts().sort_index())

SGD unweighted (argmax)      Total_cost =  1,463,385   (per readout: 6.54)
SGD unweighted (min cost)    Total_cost =  1,457,252   (per readout: 6.51)
0    221470
1      2030
2       210
3        12
4       199
Name: count, dtype: int64


In [18]:
cost_report(y_val, np.zeros_like(y_val), "Always predict 0")
cost_report(y_val, np.full_like(y_val, 4), "Always predict 4")

Always predict 0             Total_cost =  1,489,900   (per readout: 6.65)
Always predict 4             Total_cost =  2,224,862   (per readout: 9.94)


2224862